# Target-Style Alignment Fine-Tuning on Starlar Adapter

This notebook continues from the Starlar v2 fine-tuned Mistral QLoRA adapter and performs a short target-style alignment fine-tuning run.

Goal:
- Start from the Starlar v2 adapter.
- Use the target-style alignment SFT dataset prepared from Kaggle QA records.
- Teach the model to produce short, direct Turkish answers without source/citation lines.
- Evaluate later on the old best RAG contexts.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > L4 GPU seç.")

CUDA available: True
GPU: NVIDIA L4


In [3]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

train_jsonl_path = f"{processed_path}/target_style_alignment_sft_train.jsonl"
val_jsonl_path = f"{processed_path}/target_style_alignment_sft_val.jsonl"
test_jsonl_path = f"{processed_path}/target_style_alignment_sft_test.jsonl"

starlar_adapter_path = f"{models_path}/mistral_legal_qlora_starlar_v2_800steps"

target_aligned_adapter_path = f"{models_path}/mistral_legal_qlora_starlar_v2_target_aligned_200steps"

print("Train exists:", os.path.exists(train_jsonl_path))
print("Val exists:", os.path.exists(val_jsonl_path))
print("Test exists:", os.path.exists(test_jsonl_path))

print("\nStarlar adapter exists:", os.path.exists(starlar_adapter_path))
print("Starlar adapter config:", os.path.exists(f"{starlar_adapter_path}/adapter_config.json"))
print("Starlar adapter model:", os.path.exists(f"{starlar_adapter_path}/adapter_model.safetensors"))

print("\nTarget aligned adapter output:")
print(target_aligned_adapter_path)

Train exists: True
Val exists: True
Test exists: True

Starlar adapter exists: True
Starlar adapter config: True
Starlar adapter model: True

Target aligned adapter output:
/content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_target_aligned_200steps


In [4]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.9 MB/s eta 0:00:00


In [5]:
import os
import gc
import json
import math
import pandas as pd
import numpy as np
import torch

from dataclasses import dataclass
from typing import Dict, List, Any

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training
)

In [6]:
data_files = {
    "train": train_jsonl_path,
    "validation": val_jsonl_path,
    "test": test_jsonl_path
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)

print("\nTrain sample keys:")
print(dataset["train"][0].keys())

print("\nSample text:")
print(dataset["train"][0]["text"][:2000])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'question', 'answer', 'context', 'source', 'score', 'answer_context_recall', 'split_source'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['text', 'question', 'answer', 'context', 'source', 'score', 'answer_context_recall', 'split_source'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'question', 'answer', 'context', 'source', 'score', 'answer_context_recall', 'split_source'],
        num_rows: 200
    })
})

Train sample keys:
dict_keys(['text', 'question', 'answer', 'context', 'source', 'score', 'answer_context_recall', 'split_source'])

Sample text:
<s>[INST] Sen Türk hukuk soruları için çalışan dikkatli bir RAG asistanısın.
Cevabı yalnızca verilen bağlama dayanarak üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, doğrudan ve Türkçe olmalı.
Kaynak, citation, chunk id veya bağlam numarası yazma.
Sadece nihai cevabı yaz.

Bağlam:
İKİNCİ KİTAP

Soruş

In [7]:
train_df_preview = pd.DataFrame(dataset["train"])
val_df_preview = pd.DataFrame(dataset["validation"])
test_df_preview = pd.DataFrame(dataset["test"])

print("Train:", train_df_preview.shape)
print("Val:", val_df_preview.shape)
print("Test:", test_df_preview.shape)

display(train_df_preview[["question", "answer", "answer_context_recall", "score"]].head())
display(train_df_preview[["answer_context_recall"]].describe())

Train: (1500, 8)
Val: (200, 8)
Test: (200, 8)


,question,answer,answer_context_recall,score
0,Kamu davasının açılmasının ertelenmesi süresin...,Erteleme süresi içinde şüpheli kasıtlı bir suç...,0.611111,8
1,Eğer görevlendirilen avukat yargılama sırasınd...,Eğer görevlendirilen avukat yargılama sırasınd...,0.527778,8
2,"Kiraya veren, yasal hükümlere uymazsa eski kir...","Kiraya veren, yasal hükümlere uymazsa, eski ki...",0.619048,8
3,"Kovuşturmaya yer olmadığına dair karar, hangi ...","Kovuşturmaya yer olmadığına dair karar, Türk C...",0.500000,10
4,Hayatta olup olmadığı bilinmeyen bir kişinin m...,Hayatta olup olmadığı bilinmeyen bir kişinin m...,0.634146,8


,answer_context_recall
count,1500.000000
mean,0.654759
std,0.189985
min,0.350000
25%,0.500000
50%,0.636364
75%,0.789474
max,1.000000


In [8]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)
print("Truncation side:", tokenizer.truncation_side)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: </s>
Padding side: right
Truncation side: left


In [10]:
MAX_LENGTH = 1024
RESPONSE_MARKER = "[/INST]"


def tokenize_with_assistant_only_labels(example):
    text = str(example["text"])

    if RESPONSE_MARKER not in text:
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            add_special_tokens=False
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = input_ids.copy()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

    prompt_part = text.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER
    assistant_part = text[len(prompt_part):]

    prompt_ids = tokenizer(
        prompt_part,
        add_special_tokens=False
    )["input_ids"]

    assistant_ids = tokenizer(
        assistant_part,
        add_special_tokens=False
    )["input_ids"]

    total_len = len(prompt_ids) + len(assistant_ids)

    if total_len > MAX_LENGTH:
        overflow = total_len - MAX_LENGTH

        if overflow < len(prompt_ids):
            prompt_ids = prompt_ids[overflow:]
        else:
            prompt_ids = []
            assistant_ids = assistant_ids[-MAX_LENGTH:]

    input_ids = prompt_ids + assistant_ids
    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + assistant_ids.copy()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [11]:
tokenized_dataset = dataset.map(
    tokenize_with_assistant_only_labels,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing target-style alignment data"
)

print(tokenized_dataset)

sample = tokenized_dataset["train"][0]

print("input_ids len:", len(sample["input_ids"]))
print("labels len:", len(sample["labels"]))
print("loss token count:", sum([1 for x in sample["labels"] if x != -100]))

decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
print("\nDecoded input preview:")
print(decoded_input[:2000])

Tokenizing target-style alignment data:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing target-style alignment data:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing target-style alignment data:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})
input_ids len: 1024
labels len: 1024
loss token count: 91

Decoded input preview:
Suçun delilleri,

k) Şüphelinin tutuklu olup olmadığı; tutuklanmış ise, gözaltına alma ve tutuklama tarihleri ile bunların süreleri,

Gösterilir.

(4) İddianamede, yüklenen suçu oluşturan olaylar, mevcut delillerle ilişkilendirilerek açıklanır; yüklenen suçu oluşturan olaylar ve suçun delilleriyle ilgisi bulunmayan bilgilere yer verilmez.[62]

(5) İddianamenin sonuç kısmında, şüphelinin sadece aleyhine olan hususlar değil, lehine olan hususlar da ileri sürülür.

(6) İddianamenin sonuç kısmında, işlenen suç dolayısıyla ilgili kanunda öngörülen ceza ve güvenl

In [12]:
@dataclass
class CausalLMDataCollatorWithLabelPadding:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_length = max(len(f["input_ids"]) for f in features)

        input_ids_batch = []
        attention_mask_batch = []
        labels_batch = []

        pad_token_id = self.tokenizer.pad_token_id

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]
            labels = f["labels"]

            pad_length = max_length - len(input_ids)

            input_ids_batch.append(input_ids + [pad_token_id] * pad_length)
            attention_mask_batch.append(attention_mask + [0] * pad_length)
            labels_batch.append(labels + [self.label_pad_token_id] * pad_length)

        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long)
        }


data_collator = CausalLMDataCollatorWithLabelPadding(tokenizer=tokenizer)

print("Custom data collator ready.")

Custom data collator ready.


In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.config.use_cache = False

print("Base Mistral loaded in 4-bit.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base Mistral loaded in 4-bit.


In [14]:
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True
)

print("Base model prepared for k-bit training.")

Base model prepared for k-bit training.


In [15]:
model = PeftModel.from_pretrained(
    base_model,
    starlar_adapter_path,
    is_trainable=True
)

model.print_trainable_parameters()

print("Starlar adapter loaded as trainable.")

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
Starlar adapter loaded as trainable.


In [16]:
os.makedirs(target_aligned_adapter_path, exist_ok=True)

MAX_STEPS = 200
SAVE_STEPS = 50
EVAL_STEPS = 50

training_args = TrainingArguments(
    output_dir=target_aligned_adapter_path,

    max_steps=MAX_STEPS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    remove_unused_columns=False
)

print("Training args ready.")
print("Max steps:", MAX_STEPS)
print("Save every:", SAVE_STEPS)
print("Eval every:", EVAL_STEPS)
print("Learning rate:", training_args.learning_rate)
print("Output:", target_aligned_adapter_path)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args ready.
Max steps: 200
Save every: 50
Eval every: 50
Learning rate: 5e-05
Output: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_target_aligned_200steps


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.0
        )
    ]
)

print("Trainer ready.")

Trainer ready.


In [18]:
train_result = trainer.train()

print("Target-style alignment training finished.")

Step,Training Loss,Validation Loss
50,0.877499,0.834860
100,0.800066,0.791140
150,0.850846,0.772703
200,0.706539,0.768215


Target-style alignment training finished.


In [19]:
trainer.save_model(target_aligned_adapter_path)
tokenizer.save_pretrained(target_aligned_adapter_path)

print("Target-aligned adapter saved to:", target_aligned_adapter_path)

print("\nFiles in target-aligned adapter folder:")
for item in os.listdir(target_aligned_adapter_path):
    print("-", item)

Target-aligned adapter saved to: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_target_aligned_200steps

Files in target-aligned adapter folder:
- checkpoint-50
- checkpoint-100
- checkpoint-150
- checkpoint-200
- README.md
- adapter_model.safetensors
- adapter_config.json
- chat_template.jinja
- tokenizer_config.json
- tokenizer.json
- training_args.bin


In [20]:
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

for file in required_files:
    path = os.path.join(target_aligned_adapter_path, file)
    print(file, "exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size KB:", round(os.path.getsize(path) / 1024, 2))
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 4))

adapter_config.json exists: True
Size KB: 1.09
Size MB: 0.0011
adapter_model.safetensors exists: True
Size KB: 163898.67
Size MB: 160.0573


In [21]:
log_df = pd.DataFrame(trainer.state.log_history)

display(log_df.tail(30))

log_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_target_aligned_200steps_training_log.csv"

log_df.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

print("Training log saved:", log_path)

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,1.175099,2.395018,4.997050e-05,0.053333,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.976934,2.270314,4.944806e-05,0.106667,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.961600,2.004851,4.828590e-05,0.160000,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.919700,2.157134,4.651443e-05,0.213333,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.877499,2.396588,4.417999e-05,0.266667,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,0.266667,50,0.834860,170.8746,1.170,1.170,NaN,NaN,NaN,NaN,NaN
6,0.871462,2.430721,4.134369e-05,0.320000,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.889987,2.363071,3.807972e-05,0.373333,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.805888,1.997891,3.447350e-05,0.426667,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.886392,2.086080,3.061939e-05,0.480000,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Training log saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_training_log.csv


In [22]:
summary_df = pd.DataFrame([{
    "base_model": model_name,
    "starting_adapter": starlar_adapter_path,
    "output_adapter": target_aligned_adapter_path,
    "method": "QLoRA continued adapter training",
    "dataset": "target_style_alignment_sft",
    "train_file": train_jsonl_path,
    "val_file": val_jsonl_path,
    "train_rows": len(dataset["train"]),
    "val_rows": len(dataset["validation"]),
    "max_steps": MAX_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "max_length": MAX_LENGTH,
    "assistant_only_loss": True,
    "learning_rate": training_args.learning_rate,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric
}])

summary_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_target_aligned_200steps_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

print("Summary saved:", summary_path)

,base_model,starting_adapter,output_adapter,method,dataset,train_file,val_file,train_rows,val_rows,max_steps,save_steps,eval_steps,max_length,assistant_only_loss,learning_rate,gradient_accumulation_steps,per_device_train_batch_size,best_model_checkpoint,best_metric
0,mistralai/Mistral-7B-Instruct-v0.2,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,QLoRA continued adapter training,target_style_alignment_sft,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,1500,200,200,50,50,1024,True,0.00005,8,1,/content/drive/MyDrive/turkish_legal_rag/outpu...,0.768215


Summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_summary.csv


In [23]:
def extract_prompt_from_sft_text(text):
    text = str(text)

    if RESPONSE_MARKER in text:
        return text.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER

    return text


def generate_answer_only(model, tokenizer, prompt, max_new_tokens=160, max_length=1024):
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()

In [24]:
sample_text = dataset["test"][0]["text"]
sample_prompt = extract_prompt_from_sft_text(sample_text)

expected_answer = dataset["test"][0]["answer"]

generated_answer = generate_answer_only(
    model=model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=160,
    max_length=1024
)

print("PROMPT LAST PART:")
print(sample_prompt[-1500:])

print("\nEXPECTED:")
print(expected_answer)

print("\nGENERATED:")
print(generated_answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


PROMPT LAST PART:
özden geçirmekle görebileceği ayıplardan da, ancak

böyle bir ayıbın bulunmadığını ayrıca üstlenmişse sorumlu olur.

4. Gözden geçirme ve satıcıya bildirme

a. Genel olarak

MADDE 223- Alıcı, devraldığı satılanın durumunu işlerin olağan akışına göre imkân

bulunur bulunmaz gözden geçirmek ve satılanda satıcının sorumluluğunu gerektiren bir ayıp

görürse, bunu uygun bir süre içinde ona bildirmek zorundadır.

Alıcı gözden geçirmeyi ve bildirimde bulunmayı ihmal ederse, satılanı kabul etmiş

sayılır. Ancak, satılanda olağan bir gözden geçirmeyle ortaya çıkarılamayacak bir ayıp

bulunması hâlinde, bu hüküm uygulanmaz. Bu tür bir ayıbın bulunduğu sonradan anlaşılırsa,

hemen satıcıya bildirilmelidir; bildirilmezse satılan bu ayıpla birlikte kabul edilmiş sayılır.

b. Hayvan satışında

MADDE 224- Hayvan satışında satıcının sorumlu olacağı süre yazılı olarak

belirlenmemiş ve ayıp da hayvanın gebeliğine ilişkin değilse satıcı, ancak ayıbın devrin

yapıldığı veya alıcının dev

In [25]:
# Multi-sample sanity test for target-aligned adapter

def run_sanity_sample(sample_idx, max_new_tokens=220):
    sample_text = dataset["test"][sample_idx]["text"]
    sample_prompt = extract_prompt_from_sft_text(sample_text)

    expected_answer = dataset["test"][sample_idx]["answer"]

    generated_answer = generate_answer_only(
        model=model,
        tokenizer=tokenizer,
        prompt=sample_prompt,
        max_new_tokens=max_new_tokens,
        max_length=1024
    )

    print("=" * 120)
    print("SAMPLE INDEX:", sample_idx)

    print("\nQUESTION:")
    print(dataset["test"][sample_idx]["question"])

    print("\nEXPECTED:")
    print(expected_answer)

    print("\nGENERATED:")
    print(generated_answer)


for idx in range(5):
    run_sanity_sample(idx, max_new_tokens=220)

SAMPLE INDEX: 0

QUESTION:
Alıcının sözleşmeden dönme hakkını kullanması durumunda hâkim hangi kararları verebilir?

EXPECTED:
Alıcının sözleşmeden dönme hakkını kullanması durumunda, durum bunu haklı göstermiyorsa hâkim, malın onarılmasına veya satış bedelinin indirilmesine karar verebilir. Hâkimin bu kararları, adil bir çözüm sağlamak ve tarafların haklarını korumak için verilir. Hâkim, olayın koşullarını ve tarafların haklarını dikkate alarak en uygun çözümü belirler. Bu kararlar, alıcının haklarını korurken, satıcıya da adil bir davranış sergilenmesini sağlar.

GENERATED:
Alıcının sözleşmeden dönme hakkını kullanması durumunda hâkim, alıcının sözleşmeden dönme hakkını kullanmasının sonuçlarını değerlendirerek, alıcının sözleşmeden dönme hakkını kullanmasının sonuçlarının gerektirdiği şekilde kararlar verebilir. Bu kararlar, alıcının sözleşmeden dönme hakkını kullanmasının sonuçlarının gerektirdiği şekilde değerlendirilir.
SAMPLE INDEX: 1

QUESTION:
Ağaçtan yapılmış Türk bayrağı dir

In [30]:
target_alignment_decision_df = pd.DataFrame([{
    "experiment": "Target-style alignment fine-tuning on Starlar adapter",
    "starting_adapter": starlar_adapter_path,
    "output_adapter": target_aligned_adapter_path,
    "dataset": "target_style_alignment_sft",
    "max_steps": MAX_STEPS,
    "learning_rate": training_args.learning_rate,
    "best_eval_loss": trainer.state.best_metric,
    "selected_for_final": False,
    "reason": (
        "Although validation loss decreased, qualitative sanity tests showed weak answer quality. "
        "The target-aligned adapter produced generic, repetitive, and sometimes linguistically degraded Turkish answers. "
        "Therefore, it was not selected as the final generator."
    ),
    "final_decision": (
        "Keep the original Starlar fine-tuned adapter for controlled gold-context generation results, "
        "and keep the old best Base Mistral generator for the original end-to-end RAG benchmark."
    )
}])

target_alignment_decision_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_target_aligned_200steps_decision_summary.csv"

target_alignment_decision_df.to_csv(
    target_alignment_decision_path,
    index=False,
    encoding="utf-8-sig"
)

display(target_alignment_decision_df)

print("Decision summary saved:", target_alignment_decision_path)

,experiment,starting_adapter,output_adapter,dataset,max_steps,learning_rate,best_eval_loss,selected_for_final,reason,final_decision
0,Target-style alignment fine-tuning on Starlar ...,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,target_style_alignment_sft,200,0.00005,0.768215,False,"Although validation loss decreased, qualitativ...",Keep the original Starlar fine-tuned adapter f...


Decision summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_decision_summary.csv


In [31]:
files_to_check = [
    log_path,
    summary_path,
    target_alignment_decision_path,
    os.path.join(target_aligned_adapter_path, "adapter_config.json"),
    os.path.join(target_aligned_adapter_path, "adapter_model.safetensors")
]

print("FINAL CHECK")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Notebook 24 completed as ablation / not selected.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_training_log.csv
Exists: True
Size KB: 2.21
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_summary.csv
Exists: True
Size KB: 0.92
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_target_aligned_200steps_decision_summary.csv
Exists: True
Size KB: 0.88
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_target_aligned_200steps/adapter_config.json
Exists: True
Size KB: 1.09
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_ra